# Build SFINCS Model

Follows the [Deltares hydromt_sfincs build-from-script example](https://deltares.github.io/hydromt_sfincs/latest/_examples/1_build_from_script.html) step by step, adapted for Marshfield using local data.

> **Stage Contract**
>
> Requires: region setup outputs and event catalog products
>
> Produces: standard SFINCS base model
>
> Next: 05_create_scenarios.ipynb when coastal_waves is false or for comparison

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import os
os.environ.pop("DEBUG", None)

from hydromt_sfincs import SfincsModel, DATADIR
import hydromt

print("hydromt_sfincs:", __import__('hydromt_sfincs').__version__)
print("hydromt:", hydromt.__version__)

In [ ]:
import sys
from pathlib import Path

location_root = Path("../..").resolve()
repo_root = location_root.parents[1]
src_root = repo_root / "src"
sys.path.insert(0, str(src_root))
workspace = location_root

# model, scenario, storage, and catalog paths.
from sfincs_runs.notebook import load_runtime

runtime = load_runtime(location_root)
config = runtime.config
paths = runtime.paths

static_dir = runtime.static_dir
sfincs_root = runtime.sfincs_root
base_model = runtime.base_model

design_outputs = runtime.design_outputs
events_dir = runtime.events_dir
dep_dir = runtime.dep_dir
catalog_dir = runtime.catalog_dir

# Local rasters.
cudem_tif = static_dir / "cudem_mfield_mesh.tif"   # CUDEM clipped to the imported mesh footprint, EPSG:26919
worldcover_tif = static_dir / "worldcover_mfield_mesh.tif"
mesh_hdf = None
mesh_clean = static_dir / "unstructured_mfield_mesh_ugrid.nc"

# Mapping table bundled with hydromt_sfincs.
esa_mapping = Path(DATADIR) / "lulc" / "esa_worldcover_mapping.csv"

print("Location workspace:", workspace)
print("Base model root:", base_model)


## Part 1 — Build Base Model

### Step 1 · Initialize SfincsModel


In [ ]:
sf = SfincsModel(
    root=str(base_model),
    mode="w+",
    write_gis=True,
)

sf.data_catalog.from_dict({
    "cudem_elv": {
        "uri": str(cudem_tif),
        "data_type": "RasterDataset",
        "driver": {"name": "rasterio"},
    },
    "worldcover": {
        "uri": str(worldcover_tif),
        "data_type": "RasterDataset",
        "driver": {"name": "rasterio"},
    },
})

print("Catalog sources:", list(sf.data_catalog))

### Step 2 · Create Grid


In [ ]:
from types import SimpleNamespace
from shapely.geometry import LineString, Polygon

# create domains, handoff points, observations, and physics checks.
from sfincs_runs.build_base.build_graph import derive_east_boundary_from_ugrid, load_unstructured_mesh_from_hdf
# store the derived mesh boundary polygon for model setup.
from sfincs_runs.build_base.geometry import write_geom

def load_notebook_mesh():
    if mesh_hdf is not None and mesh_hdf.exists():
        return load_unstructured_mesh_from_hdf(mesh_hdf, clean_mesh_path=mesh_clean)

    perimeter_xy, east_line_xy, east_sample_xy = derive_east_boundary_from_ugrid(mesh_clean)
    return SimpleNamespace(
        perimeter_xy=perimeter_xy,
        east_boundary_line_xy=east_line_xy,
        east_boundary_sample_xy=east_sample_xy,
        east_boundary_faces=np.arange(east_sample_xy.shape[0], dtype=int),
        report=SimpleNamespace(crs="EPSG:26919"),
    )

mesh = load_notebook_mesh()

mesh_region_geojson = static_dir / "marshfield_mesh_region.geojson"
if mesh_hdf is not None and mesh_hdf.exists():
    mesh_region = gpd.GeoDataFrame(
        {"name": ["marshfield_mesh"]},
        geometry=[Polygon(mesh.perimeter_xy)],
        crs=mesh.report.crs,
    )
    mesh_region.to_file(mesh_region_geojson, driver="GeoJSON")
else:
    write_geom(mesh.perimeter_xy, mesh.report.crs, mesh_region_geojson)
    mesh_region = gpd.read_file(mesh_region_geojson)

# East-edge buffer: a thin polygon around the imported east-boundary polyline,
# wide enough to capture exactly one 90-m grid cell along the mesh east edge.
east_buffer_geojson = static_dir / "marshfield_east_boundary_buffer.geojson"
east_buffer = gpd.GeoDataFrame(
    {"name": ["east_edge"]},
    geometry=[LineString(mesh.east_boundary_line_xy).buffer(180.0)],
        crs=mesh.report.crs,
)
east_buffer.to_file(east_buffer_geojson, driver="GeoJSON")

print(f"Mesh region bounds (UTM 26919): {mesh_region.total_bounds}")

sf.grid.create_from_region(
    region={"geom": str(mesh_region_geojson)},
    res=30, # total terrain resolution
    rotated=True,
    crs="utm",
)

print("Grid CRS:", sf.crs)
print("Grid res:", sf.grid.res)

sf.plot_basemap(variable="grid", plot_region=True, bmap="sat", zoomlevel=12)

### Step 3 · Add Elevation

Using CUDEM (Continuously Updated Digital Elevation Model) which seamlessly merges
onshore topography and offshore bathymetry. Single source — no GEBCO fill needed.


In [ ]:
elevation_list = [
    {"elevation": "cudem_elv"}, # CUDEM: full topo+bathy, no zmin filter needed
]

sf.elevation.create(elevation_list=elevation_list, buffer_cells=1)

sf.plot_basemap(variable="dep", plot_region=True, bmap="sat", zoomlevel=12)

### Step 4 · Create Active / Inactive Cell Mask

The active mask is conformed to the imported mesh perimeter via `include_polygon`,
not to an elevation threshold. The mesh was designed to stop just offshore, so
every cell inside it is physically meaningful — using a `zmin` cutoff would
deactivate the deep-ocean strip along the east edge and leave the water-level
boundary with nowhere to land.


In [ ]:
sf.mask.create_active(
    include_polygon=str(mesh_region_geojson),
    reset_mask=True,
)

sf.plot_basemap(variable="mask", plot_region=False, plot_bounds=True, bmap="sat", zoomlevel=12)

### Step 5 · Create Boundary Cells

**Water-level boundary** (mask = 2): restricted to active cells that fall inside the
east-edge buffer polygon, so the open-ocean forcing lines up with the mesh east edge
exactly where the imported forcing polyline sits. All other mesh edges (inland/N/S/W)
stay closed by default.


In [ ]:
sf.mask.create_boundary(
    btype="waterlevel",
    include_polygon=str(east_buffer_geojson),
    reset_bounds=True,
)

msk = sf.mask.data
n_wl = int((msk["mask"].values == 2).sum())
n_active = int((msk["mask"].values == 1).sum())
print(f"Active cells: {n_active}  |  Water-level boundary cells: {n_wl}")

sf.plot_basemap(variable="mask", plot_region=False, plot_bounds=True, bmap="sat", zoomlevel=12)

### Step 6 · Add Roughness

Using ESA WorldCover 2020 (local raster) reclassified to Manning's n via the bundled mapping table.


In [ ]:
roughness_list = [
    {"lulc": "worldcover", "reclass_table": str(esa_mapping)},
]

sf.roughness.create(
    roughness_list=roughness_list,
    manning_land=0.04,   # fallback for unclassified land cells
    manning_sea=0.02,    # open-water cells
    rgh_lev_land=0,
)

sf.plot_basemap(variable="manning", plot_bounds=False, bmap="sat", zoomlevel=12)

### Step 7 · Create Subgrid Tables

Subgrid tables allow a coarse (90 m) grid to capture fine-scale bathymetric and roughness variability. 
This is computationally expensive to build but greatly improves accuracy without increasing runtime cost.


In [ ]:
sf.subgrid.create(
    elevation_list=[
        {"elevation": "cudem_elv"},
    ],
    roughness_list=[
        {"lulc": "worldcover", "reclass_table": str(esa_mapping)},
    ],
    nr_subgrid_pixels=6, # 6 subgrid pixels per model cell → ~5 m effective resolution
    write_dep_tif=True,
    write_man_tif=True,
)

### Step 8 · Write Base Model


In [ ]:
# add infiltration/soil inputs used by SFINCS rainfall runs.
from sfincs_runs.hydrology import setup_hydromt_infiltration

infiltration_summary = setup_hydromt_infiltration(sf, config, paths, datadir=DATADIR)
print("Hydrology enabled:", infiltration_summary["enabled"])
print("Hydrologic event drivers:", infiltration_summary["drivers"])
print("Infiltration method:", infiltration_summary["method"])
print("Infiltration files written:", infiltration_summary["written"])

sf.config.update({
    "bndfile": "sfincs.bnd",
    "epsg": 26919,
    "storevel": 1,
})

sf.write()
print("Base model written to:", base_model)


## Coastal boundary showing in blue


In [ ]:
# Full basemap of the completed base model
sf.plot_basemap(
    variable="dep",
    plot_geoms=True,
    plot_bounds=True,
    bmap="sat",
    zoomlevel=12,
)